# IT Incidents — Dashboard

This notebook is a standalone dashboard, built with pandas and matplotlib, showing basic statistics and charts about the incidents recorded by the Django application. It is a complementary artifact, separate from the in-app dashboard view still pending inside the `incidents` Django app. It reads directly from the project's SQLite database (`src/db.sqlite3`) through a read-only connection, without importing Django or its ORM, so this module stays fully decoupled from the web application.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DB_PATH = Path("src/db.sqlite3").resolve()
connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

incidents = pd.read_sql_query(
    "SELECT title, equipment, priority, status, is_archived, date FROM incidents_incident",
    connection,
)
connection.close()

PRIORITY_ORDER = ["low", "medium", "high"]
STATUS_ORDER = ["open", "in_progress", "closed"]

incidents.head()

## Total incidents

Total number of incidents ever recorded in the system, regardless of status or archive state.

In [ ]:
len(incidents)

## Open incidents

Incidents that are not closed and have not been archived, whatever their current status (open or in progress).

In [ ]:
open_incidents = incidents[
    (incidents["status"] != "closed") & (incidents["is_archived"] == 0)
]
len(open_incidents)

## Closed incidents

Incidents whose status has been set to closed.

In [ ]:
closed_incidents = incidents[incidents["status"] == "closed"]
len(closed_incidents)

## High-priority incidents

Incidents currently marked with high priority.

In [ ]:
high_priority_incidents = incidents[incidents["priority"] == "high"]
len(high_priority_incidents)

## Incidents by priority

Bar chart with the number of incidents for each priority level.

In [ ]:
priority_counts = (
    incidents["priority"].value_counts().reindex(PRIORITY_ORDER, fill_value=0)
)

fig, ax = plt.subplots()
ax.bar(priority_counts.index, priority_counts.values)
ax.set_title("Incidents by priority")
ax.set_xlabel("Priority")
ax.set_ylabel("Number of incidents")
plt.show()

## Incidents by status

Bar chart with the number of incidents for each status.

In [ ]:
status_counts = incidents["status"].value_counts().reindex(STATUS_ORDER, fill_value=0)

fig, ax = plt.subplots()
ax.bar(status_counts.index, status_counts.values)
ax.set_title("Incidents by status")
ax.set_xlabel("Status")
ax.set_ylabel("Number of incidents")
plt.show()

## Incident evolution over time

Line chart with the number of incidents created per time period. `plot_evolution` takes the time bucket and an optional date range, so re-running the call cell below with different arguments changes the view — see the examples underneath it.

In [ ]:
def plot_evolution(freq: str = "ME", start: str | None = None, end: str | None = None) -> None:
    """Plot the number of incidents created per time period.

    :param freq: Pandas offset alias for the time bucket (e.g. "D" for
        daily, "W" for weekly, "ME" for monthly, "YE" for yearly).
    :param start: Optional inclusive lower bound (e.g. "2026-01-01") to
        restrict the date range plotted.
    :param end: Optional inclusive upper bound to restrict the date range
        plotted.
    :return: None. Displays a matplotlib line chart.
    """
    created_at = pd.to_datetime(incidents["date"])
    counts_by_date = pd.Series(1, index=created_at)

    if start is not None:
        counts_by_date = counts_by_date[counts_by_date.index >= start]
    if end is not None:
        counts_by_date = counts_by_date[counts_by_date.index <= end]

    counts_by_period = counts_by_date.resample(freq).sum()

    fig, ax = plt.subplots()
    ax.plot(counts_by_period.index, counts_by_period.values, marker="o")
    ax.set_title("Incident evolution")
    ax.set_xlabel("Date")
    ax.set_ylabel("Number of incidents")
    fig.autofmt_xdate()
    plt.show()

In [ ]:
plot_evolution()

To explore other time scales or a specific date range, edit and re-run the call above, e.g.:

```python
plot_evolution(freq="W")
plot_evolution(freq="D", start="2026-01-01", end="2026-06-30")
plot_evolution(freq="YE")
```

Live, widget-based controls (e.g. `ipywidgets` dropdowns/date pickers) would let a viewer change these without editing code, but that's a possible future improvement, not implemented here.